# Themenanalyse der Kommentare mit Sprachmodell-Embeddings

Dieses Notebook analysiert Kommentar-Themen mit einem echten Sprachmodell statt mit Zero-Shot-Labels.

Ansatz:
- Kommentare werden semantisch mit einem Sentence-Embedding-Modell eingebettet
- die Embeddings werden geclustert
- Topics werden datengetrieben mit einer c-TF-IDF-ähnlichen Keyword-Gewichtung pro Cluster benannt
- corpus-spezifische Stopwords werden nach Schofield, Magnusson & Mimno (2017) erst nach der Topic-Inferenz aus der Label-Anzeige gefiltert
- die ausgeschlossenen Label-Terme werden mit Frequenz- und Topic-Spread-Diagnostik als Audit-Datei exportiert
- danach werden KI- und Real-Kommentare sowie die Zusammenhänge mit Engagement ausgewertet


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
import torch
from transformers import AutoModel, AutoTokenizer



In [2]:
BASE_DIR = Path.cwd().resolve().parents[1]
COMMENTS_PATH = BASE_DIR / "data" / "01_raw" / "comments_metadata" / "02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv"
VIDEOS_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
AI_VIDEOS_RAW_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_AI_TIKTOK_VIDEOS_DATASET_2025.csv"
REAL_VIDEOS_RAW_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
OUTPUT_DIR = BASE_DIR / "comments" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "comment_text"
FILTER_ENGLISH_ONLY = True
KEEP_REPLIES = True
BATCH_SIZE = 64
RANDOM_STATE = 42
SHOW_PROGRESS = False
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

COMMENT_RESULTS_CSV = OUTPUT_DIR / "04_comment_topics_results.csv"
VIDEO_LEVEL_CSV = OUTPUT_DIR / "04_comment_topics_video_level.csv"
TOPIC_ENGAGEMENT_CSV = OUTPUT_DIR / "04_comment_topics_engagement_relations.csv"
TOPIC_KEYWORDS_CSV = OUTPUT_DIR / "04_comment_topics_keywords.csv"
LABEL_STOPWORD_CANDIDATES_CSV = OUTPUT_DIR / "04_comment_topics_label_stopword_candidates.csv"
LABEL_STOPWORDS_CSV = OUTPUT_DIR / "04_comment_topics_label_stopwords.csv"
TOPIC_TEXT_EXCLUDED_CSV = OUTPUT_DIR / "04_comment_topics_excluded_empty_topic_text.csv"
MISSING_VIDEO_METADATA_CSV = OUTPUT_DIR / "04_comment_topics_missing_video_metadata.csv"
UNRESOLVED_VIDEO_METADATA_CSV = OUTPUT_DIR / "04_comment_topics_unresolved_video_metadata.csv"
MISSING_ENGAGEMENT_EXCLUDED_CSV = OUTPUT_DIR / "04_comment_topics_excluded_missing_engagement.csv"
ENGAGEMENT_SOURCE_CSV = OUTPUT_DIR / "04_comment_topics_engagement_sources.csv"

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 80)

config_overview = pd.DataFrame([
    ("Kommentare", COMMENTS_PATH.name),
    ("Output", str(OUTPUT_DIR.relative_to(BASE_DIR))),
    ("Embedding-Modell", EMBEDDING_MODEL_NAME),
    ("Progressbar anzeigen", SHOW_PROGRESS),
    ("Englischfilter", FILTER_ENGLISH_ONLY),
    ("Replies behalten", KEEP_REPLIES),
], columns=["Parameter", "Wert"])
display(config_overview)


,Parameter,Wert
0,Kommentare,02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv
1,Output,comments/results
2,Embedding-Modell,sentence-transformers/all-MiniLM-L6-v2
3,Progressbar anzeigen,False
4,Englischfilter,True
5,Replies behalten,True


In [ ]:
keep_cols = [
    "comment_id",
    "video_id",
    "influencer_type",
    "comment_language",
    "is_reply_to_comment",
    "comment_like_count",
    TEXT_COLUMN,
]
comments_df = pd.read_csv(COMMENTS_PATH, usecols=keep_cols).copy()
initial_n = len(comments_df)
comments_df["video_id"] = comments_df["video_id"].astype("string").str.strip()

video_meta = pd.read_csv(VIDEOS_PATH, usecols=["video_id", "video_engagement_rate"]).drop_duplicates(subset="video_id")
video_meta["video_id"] = video_meta["video_id"].astype("string").str.strip()
video_meta["engagement_source"] = "combined_video_dataset"

raw_video_meta_frames = []
for raw_path, raw_type in [(AI_VIDEOS_RAW_PATH, "ai_raw_video_dataset"), (REAL_VIDEOS_RAW_PATH, "real_raw_video_dataset")]:
    raw_meta = pd.read_csv(raw_path, usecols=["id", "likes", "comments", "shares", "plays"])
    raw_meta = raw_meta.rename(columns={"id": "video_id"})
    raw_meta["video_id"] = raw_meta["video_id"].astype("string").str.strip()
    raw_meta["video_engagement_rate"] = (
        raw_meta["likes"].fillna(0) + raw_meta["comments"].fillna(0) + raw_meta["shares"].fillna(0)
    ) / raw_meta["plays"].replace(0, np.nan)
    raw_meta["engagement_source"] = raw_type
    raw_video_meta_frames.append(raw_meta[["video_id", "video_engagement_rate", "engagement_source"]])
raw_video_meta = pd.concat(raw_video_meta_frames, ignore_index=True).drop_duplicates(subset="video_id")

video_meta_complete = pd.concat([video_meta, raw_video_meta], ignore_index=True)
video_meta_complete = video_meta_complete.dropna(subset=["video_engagement_rate"])
video_meta_complete = video_meta_complete.drop_duplicates(subset="video_id", keep="first")
comments_df = comments_df.merge(video_meta_complete, on="video_id", how="left", indicator="video_metadata_merge")
merged_n = len(comments_df)
comments_df = comments_df.drop(columns="video_metadata_merge")

comments_df = comments_df.drop_duplicates(subset="comment_id").copy()
dedup_n = len(comments_df)
comments_df[TEXT_COLUMN] = (
    comments_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
comments_df = comments_df[comments_df[TEXT_COLUMN] != ""].copy()
after_text_clean_n = len(comments_df)
if FILTER_ENGLISH_ONLY:
    comments_df = comments_df[comments_df["comment_language"].fillna("").str.lower().eq("en")].copy()
after_language_filter_n = len(comments_df)
if not KEEP_REPLIES:
    comments_df = comments_df[comments_df["is_reply_to_comment"].fillna("no").str.lower().ne("yes")].copy()
after_reply_filter_n = len(comments_df)

def prepare_topic_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", lambda m: f" {m.group(1)} ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

comments_df["topic_text"] = comments_df[TEXT_COLUMN].astype(str).apply(prepare_topic_text)
empty_topic_text_mask = comments_df["topic_text"].eq("")
excluded_empty_topic_text_df = comments_df.loc[
    empty_topic_text_mask,
    ["comment_id", "video_id", "influencer_type", "comment_language", "is_reply_to_comment", TEXT_COLUMN, "topic_text"],
].copy()
comments_df = comments_df[~empty_topic_text_mask].copy()
after_topic_text_prep_n = len(comments_df)

missing_video_metadata_df = (
    comments_df[comments_df["video_engagement_rate"].isna()]
    .groupby(["video_id", "influencer_type"], dropna=False)
    .agg(
        comment_count=("comment_id", "count"),
        example_comment=(TEXT_COLUMN, "first"),
    )
    .reset_index()
    .sort_values("comment_count", ascending=False)
)
unresolved_video_metadata_df = missing_video_metadata_df.copy()
excluded_missing_engagement_df = comments_df[comments_df["video_engagement_rate"].isna()].copy()
comments_df = comments_df[comments_df["video_engagement_rate"].notna()].copy()
after_engagement_filter_n = len(comments_df)
engagement_source_video_df = (
    comments_df.groupby(["video_id", "influencer_type", "engagement_source", "video_engagement_rate"], dropna=False)
    .size()
    .rename("comment_count")
    .reset_index()
    .sort_values(["engagement_source", "comment_count"], ascending=[True, False])
)

comments_df["Typ"] = comments_df["influencer_type"].map({"ai": "KI", "real": "Real"}).fillna(comments_df["influencer_type"])
comments_df["comment_length_chars"] = comments_df[TEXT_COLUMN].str.len()
comments_df["comment_length_words"] = comments_df[TEXT_COLUMN].str.split().str.len()
missing_engagement_before_filter = len(excluded_missing_engagement_df)
missing_engagement_video_n = excluded_missing_engagement_df["video_id"].nunique()
missing_engagement = comments_df["video_engagement_rate"].isna().sum()

prep_overview = pd.DataFrame([
    ("Ausgangsdatensatz", initial_n),
    ("Nach Engagement-Merge", merged_n),
    ("Nach Entfernen doppelter comment_id", dedup_n),
    ("Nach Entfernen leerer Kommentare", after_text_clean_n),
    ("Nach Sprachfilter (nur Englisch)" if FILTER_ENGLISH_ONLY else "Ohne Sprachfilter", after_language_filter_n),
    ("Replies beibehalten" if KEEP_REPLIES else "Nach Reply-Filter", after_reply_filter_n),
    ("Nach Topic-Textbereinigung", after_topic_text_prep_n),
    ("Ausgeschlossen: keine Engagement-Rate", missing_engagement_before_filter),
    ("Videos ohne gematchte Engagement-Rate", missing_engagement_video_n),
    ("Finale Topic-Analyse mit Engagement-Rate", after_engagement_filter_n),
], columns=["Schritt", "Kommentare"])

analysis_overview = pd.DataFrame([
    ("Finale Kommentare", after_engagement_filter_n),
    ("Ausgeschlossen: fehlende Engagement-Rate", missing_engagement_before_filter),
    ("Videos ohne Engagement-Match", missing_engagement_video_n),
    ("Ausgeschlossen: leerer Topic-Text", len(excluded_empty_topic_text_df)),
], columns=["Kennzahl", "Wert"])

engagement_source_overview = (
    comments_df["engagement_source"]
    .fillna("missing")
    .value_counts()
    .rename_axis("engagement_source")
    .reset_index(name="comments")
)

sample_comments_preview = (
    comments_df[["comment_id", TEXT_COLUMN, "topic_text", "Typ"]]
    .head(3)
    .copy()
)

print("Preprocessing-Überblick")
display(prep_overview)
print("Kompakte Kontrollkennzahlen")
display(analysis_overview)
print("Quelle der Engagement-Rate")
display(engagement_source_overview)

if not missing_video_metadata_df.empty:
    print("Top 5 Videos ohne gematchte Engagement-Rate")
    display(missing_video_metadata_df.head(5))
else:
    print("Keine Videos ohne gematchte Engagement-Rate.")

print("Beispielkommentare nach Textbereinigung")
display(sample_comments_preview)


Preprocessing-Überblick


,Schritt,Kommentare
0,Ausgangsdatensatz,80817
1,Nach Engagement-Merge,80817
2,Nach Entfernen doppelter comment_id,80817
3,Nach Entfernen leerer Kommentare,80497
4,Nach Sprachfilter (nur Englisch),39615
5,Replies beibehalten,39615
6,Nach Topic-Textbereinigung,39593
7,Ausgeschlossen: keine Engagement-Rate,6937
8,Videos ohne gematchte Engagement-Rate,83
9,Finale Topic-Analyse mit Engagement-Rate,32656


Kompakte Kontrollkennzahlen


,Kennzahl,Wert
0,Finale Kommentare,32656
1,Ausgeschlossen: fehlende Engagement-Rate,6937
2,Videos ohne Engagement-Match,83
3,Ausgeschlossen: leerer Topic-Text,22


Quelle der Engagement-Rate


,engagement_source,comments
0,combined_video_dataset,30956
1,ai_raw_video_dataset,1462
2,real_raw_video_dataset,238


Top 5 Videos ohne gematchte Engagement-Rate


,video_id,influencer_type,comment_count,example_comment
3,7208351443193974062,real,1786,If Niall Horan looked at me like that I’d forget how to breathe
6,7371613331704139040,real,1412,Julia Zelg for me. I couldn’t recognize her when I saw one of her recent ig ...
2,7040984071631228206,real,853,So y’all rich rich. Huh. Are you looking to adopt a 19 year old?
1,7014140703680826629,real,697,I don’t have a nice side profile like u girl :/
15,7538581166312426774,real,374,"go to David's salon, your $3 would be so much worth it"


Beispielkommentare nach Textbereinigung


,comment_id,comment_text,topic_text,Typ
1178,7541804977875550983,Wow I liked 💗,Wow I liked 💗,Real
1192,7543754834214945557,Need this therapy too✨,Need this therapy too✨,Real
1197,7572978399367676686,and the little sister too,and the little sister too,Real


In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Topic-Embeddings: Device={device}, Modell={EMBEDDING_MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
model = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME).to(device)
model.eval()

texts = comments_df["topic_text"].tolist()
print(f"Topic-Modelling startet: {len(texts):,} Kommentare")
embeddings = []
for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding-Kommentare", leave=False, disable=not SHOW_PROGRESS):
    batch = texts[start:start + BATCH_SIZE]
    encoded = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        outputs = model(**encoded)
        pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
    embeddings.append(pooled.cpu().numpy())

embedding_matrix = normalize(np.vstack(embeddings))
auto_n_topics = max(2, round(np.log2(len(comments_df))))
print(f"Automatisch gewählte Topic-Anzahl: {auto_n_topics}")
cluster_model = MiniBatchKMeans(n_clusters=auto_n_topics, random_state=RANDOM_STATE, batch_size=2048, n_init=20)
comments_df["topic_id"] = cluster_model.fit_predict(embedding_matrix)

topic_documents = comments_df.groupby("topic_id")["topic_text"].apply(lambda texts: " ".join(texts.fillna("").astype(str))).sort_index()

# Corpus diagnostic for transparent post-hoc label stopword selection.
# Following Schofield, Magnusson & Mimno (2017), these terms are not removed before topic inference.
# They are inspected and filtered only when displaying topic labels.
label_term_vectorizer = CountVectorizer(
    stop_words=None,
    ngram_range=(1, 1),
    min_df=5,
    max_df=1.0,
    token_pattern=r"(?u)\b[a-z][a-z]+\b",
)
comment_term_counts = label_term_vectorizer.fit_transform(comments_df["topic_text"].fillna("").astype(str))
topic_unigram_counts = label_term_vectorizer.transform(topic_documents)
label_terms = np.array(label_term_vectorizer.get_feature_names_out())
term_corpus_count = np.asarray(comment_term_counts.sum(axis=0)).ravel()
term_comment_doc_freq = np.asarray((comment_term_counts > 0).sum(axis=0)).ravel()
term_topic_presence = np.asarray((topic_unigram_counts > 0).sum(axis=0)).ravel()
term_topic_count_array = np.asarray(topic_unigram_counts.toarray())
term_max_topic_share = term_topic_count_array.max(axis=0) / np.maximum(term_corpus_count, 1)

label_term_diagnostic_df = pd.DataFrame({
    "term": label_terms,
    "corpus_count": term_corpus_count,
    "comment_doc_freq": term_comment_doc_freq,
    "topic_presence_n": term_topic_presence,
    "topic_presence_share": term_topic_presence / len(topic_documents),
    "max_topic_share": term_max_topic_share,
})
label_term_diagnostic_df["is_standard_english_stopword"] = label_term_diagnostic_df["term"].isin(ENGLISH_STOP_WORDS)

label_stopword_candidates_df = (
    label_term_diagnostic_df[
        (~label_term_diagnostic_df["is_standard_english_stopword"])
        & (label_term_diagnostic_df["corpus_count"] >= 100)
        & (label_term_diagnostic_df["topic_presence_share"] >= 0.80)
        & (label_term_diagnostic_df["max_topic_share"] <= 0.35)
    ]
    .sort_values(["topic_presence_share", "corpus_count"], ascending=[False, False])
    .reset_index(drop=True)
)

# Finale korpusspezifische Label-Stopwords, ausgewählt nach Prüfung der obigen Diagnosetabelle.
# Die Kategorien dokumentieren, warum ein Begriff in diesem Korpus nicht als Topic-Label geeignet ist.
CORPUS_LABEL_STOPWORDS_BY_REASON = {
    "contraction_or_tokenization_artifact": {
        "don", "dont", "doesn", "didn", "isn", "aren", "arent", "wasn", "weren", "cant",
        "couldn", "wouldn", "shouldn", "haven", "hasn", "hadn", "ve", "ll", "im", "ive", "id",
        "ur", "youre", "thats", "theres", "theyre", "shes", "aint",
    },
    "interaction_or_reaction_marker": {
        "lol", "lmao", "lmfao", "haha", "hahaha", "omg", "omgg", "omggg", "bruh", "bro",
        "nah", "yeah", "yep", "yes", "hi", "hey", "hello", "hii", "hiii", "oh", "wow",
        "ok", "okay", "pls", "please", "guys", "yall", "ya", "ily",
    },
    "generic_comment_discourse_marker": {
        "like", "just", "really", "actually", "literally", "right", "know", "think", "look", "looks",
        "thing", "things", "something", "someone", "people", "person", "video", "comment", "comments",
        "tiktok", "fyp", "u",
    },
    "thread_position_marker": {"early", "first", "wait"},
}
CORPUS_LABEL_STOPWORDS = set().union(*CORPUS_LABEL_STOPWORDS_BY_REASON.values())
LABEL_DISPLAY_STOPWORDS = set(ENGLISH_STOP_WORDS).union(CORPUS_LABEL_STOPWORDS)

stopword_reason_map = {
    term: reason
    for reason, terms in CORPUS_LABEL_STOPWORDS_BY_REASON.items()
    for term in terms
}
label_stopword_audit_df = (
    label_term_diagnostic_df[label_term_diagnostic_df["term"].isin(CORPUS_LABEL_STOPWORDS)]
    .assign(reason=lambda df: df["term"].map(stopword_reason_map))
    .sort_values(["reason", "topic_presence_n", "corpus_count"], ascending=[True, False, False])
    .reset_index(drop=True)
)
missing_label_stopwords = sorted(CORPUS_LABEL_STOPWORDS - set(label_term_diagnostic_df["term"]))
if missing_label_stopwords:
    print(f"Label-Stopwords ohne Vorkommen im aktuellen Korpus: {len(missing_label_stopwords)}")

stopword_overview = pd.DataFrame([
    ("Kandidaten", len(label_stopword_candidates_df)),
    ("ausgewählte Label-Stopwords mit Vorkommen", len(label_stopword_audit_df)),
    ("ausgewählte Label-Stopwords ohne Vorkommen", len(missing_label_stopwords)),
], columns=["Kennzahl", "Wert"])
print("Label-Stopword-Diagnostik")
display(stopword_overview)
if not label_stopword_candidates_df.empty:
    display(label_stopword_candidates_df[["term", "corpus_count", "comment_doc_freq", "topic_presence_share", "max_topic_share"]].head(10))

keyword_vectorizer = CountVectorizer(
    stop_words=None,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.6,
    max_features=10000,
    token_pattern=r"(?u)\b[a-z][a-z]+\b",
)
topic_term_counts = keyword_vectorizer.fit_transform(topic_documents)
feature_names = np.array(keyword_vectorizer.get_feature_names_out())
topic_term_tf = normalize(topic_term_counts, norm="l1", axis=1)
topic_term_document_frequency = np.asarray((topic_term_counts > 0).sum(axis=0)).ravel()
topic_term_idf = np.log((1 + topic_term_counts.shape[0]) / (1 + topic_term_document_frequency)) + 1
topic_term_scores = topic_term_tf.multiply(topic_term_idf).tocsr()


def is_usable_keyword(term, selected_terms):
    tokens = term.split()
    if any(token in LABEL_DISPLAY_STOPWORDS or len(token) < 3 for token in tokens):
        return False
    if len(set(tokens)) != len(tokens):
        return False
    return all(not set(tokens).intersection(existing.split()) for existing in selected_terms)


keyword_rows = []
topic_label_map = {}
for row_index, topic_id in enumerate(topic_documents.index):
    topic_mask = (comments_df["topic_id"] == topic_id).to_numpy()
    topic_size = int(topic_mask.sum())
    topic_scores = topic_term_scores[row_index].toarray().ravel()
    keywords = []
    for feature_index in topic_scores.argsort()[::-1][:150]:
        if topic_scores[feature_index] <= 0:
            break
        keyword = feature_names[feature_index]
        if is_usable_keyword(keyword, keywords):
            keywords.append(keyword)
        if len(keywords) >= 6:
            break
    if not keywords:
        keywords = [f"topic_{topic_id}"]
    topic_label = " | ".join(keywords[:3])
    topic_label_map[topic_id] = topic_label
    keyword_rows.append({
        "topic_id": topic_id,
        "topic_label": topic_label,
        "keywords": ", ".join(keywords),
        "comment_count": topic_size,
    })

topic_keywords_df = pd.DataFrame(keyword_rows).sort_values("comment_count", ascending=False).reset_index(drop=True)
comments_df["topic_label"] = comments_df["topic_id"].map(topic_label_map)


def dominant_value(series):
    mode = series.mode(dropna=True)
    return mode.iloc[0] if not mode.empty else np.nan


video_level_df = (
    comments_df.groupby(["video_id", "Typ", "video_engagement_rate"], dropna=False)
    .agg(
        comment_count=("comment_id", "count"),
        topic_unique_labels=("topic_id", "nunique"),
        dominant_topic_id=("topic_id", dominant_value),
        dominant_topic_label=("topic_label", dominant_value),
    )
    .reset_index()
)

video_topic_share_df = (
    comments_df.groupby(["video_id", "topic_id", "topic_label"]).size().rename("topic_comment_count").reset_index()
)
video_comment_totals = comments_df.groupby("video_id").size().rename("video_comment_count").reset_index()
video_topic_share_df = video_topic_share_df.merge(video_comment_totals, on="video_id", how="left")
video_topic_share_df = video_topic_share_df.merge(
    video_level_df[["video_id", "Typ", "video_engagement_rate"]],
    on="video_id",
    how="left",
)
video_topic_share_df["topic_share"] = video_topic_share_df["topic_comment_count"] / video_topic_share_df["video_comment_count"]

engagement_rows = []
for (topic_id, topic_label), topic_slice in video_topic_share_df.groupby(["topic_id", "topic_label"]):
    corr_df = topic_slice.dropna(subset=["topic_share", "video_engagement_rate"])
    if len(corr_df) >= 3 and corr_df["topic_share"].nunique() > 1 and corr_df["video_engagement_rate"].nunique() > 1:
        rho, p_value = spearmanr(corr_df["topic_share"], corr_df["video_engagement_rate"], nan_policy="omit")
    else:
        rho, p_value = np.nan, np.nan
    engagement_rows.append({
        "topic_id": topic_id,
        "topic_label": topic_label,
        "rho": rho,
        "p": p_value,
        "signifikant": bool(pd.notna(p_value) and p_value < 0.05),
        "videos_n": int(corr_df["video_id"].nunique()),
    })

topic_engagement_df = pd.DataFrame(engagement_rows).sort_values(["signifikant", "rho"], ascending=[False, False]).reset_index(drop=True)

comments_df["topic_comment_share"] = comments_df.groupby("topic_id")["topic_id"].transform("size") / len(comments_df)
topic_keywords_df = topic_keywords_df.merge(
    comments_df.groupby("topic_id")["topic_comment_share"].first().rename("comment_share").reset_index(),
    on="topic_id",
    how="left",
)

topic_overview = topic_keywords_df[["topic_id", "topic_label", "keywords", "comment_count", "comment_share"]].copy()
topic_overview["comment_share"] = topic_overview["comment_share"].round(3)
engagement_overview = (
    topic_engagement_df
    .assign(abs_rho=lambda df: df["rho"].abs())
    .sort_values("abs_rho", ascending=False)
    .drop(columns="abs_rho")
    .head(8)
)
comment_topic_preview = comments_df[["comment_id", TEXT_COLUMN, "topic_id", "topic_label"]].head(5)
video_level_preview = video_level_df[["video_id", "Typ", "comment_count", "topic_unique_labels", "dominant_topic_label", "video_engagement_rate"]].head(5)

print("Topic-Überblick: größte Topics")
display(topic_overview.head(12))
print("Stärkste Topic-Engagement-Zusammenhänge")
display(engagement_overview)
print("Beispielhafte zugewiesene Kommentar-Topics")
display(comment_topic_preview)
print("Beispielhafte Video-Level-Aggregation")
display(video_level_preview)



Topic-Embeddings: Device=mps, Modell=sentence-transformers/all-MiniLM-L6-v2
Topic-Modelling startet: 32,656 Kommentare
Automatisch gewählte Topic-Anzahl: 15


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/hannahernstberger/Documents/Master_/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hannahernstberger/Documents/Master_/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hannahernstberger/Documents/Master_/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/hannahernstberger/Documents/Master_/venv/lib/python3.14/site-packages/sklearn/cluster/_kmeans.py:243: RuntimeWarning: divide by zero encountered in matmul

Label-Stopwords ohne Vorkommen im aktuellen Korpus: 2
Label-Stopword-Diagnostik


,Kennzahl,Wert
0,Kandidaten,154
1,ausgewählte Label-Stopwords mit Vorkommen,80
2,ausgewählte Label-Stopwords ohne Vorkommen,2


,term,corpus_count,comment_doc_freq,topic_presence_share,max_topic_share
0,like,2728,2495,1.0,0.176686
1,just,1806,1695,1.0,0.178848
2,love,1341,1283,1.0,0.261745
3,don,1335,1209,1.0,0.206742
4,people,1082,977,1.0,0.280037
5,girl,987,971,1.0,0.286727
6,think,824,787,1.0,0.268204
7,good,807,771,1.0,0.206939
8,look,794,767,1.0,0.308564
9,know,775,737,1.0,0.145806


...

,topic_id,topic_label,rho,p,signifikant,videos_n
9,2,robot | cgi | generated,-0.434378,1.136879e-07,True,137
8,14,america | europe | world country,-0.396298,1.736554e-04,True,85
7,13,cgi | animated | edited,-0.386118,7.585986e-07,True,154
6,0,gay | animated | steal,-0.299178,5.352457e-08,True,318
5,6,robot | sophia | polar,-0.297851,2.997948e-05,True,190
4,10,philippines | notary | world country,-0.284508,3.527196e-02,True,55
3,11,detroit | slay | twerk,-0.251074,4.295108e-07,True,395
2,5,almond | pain | lyme,-0.227955,6.394372e-05,True,302


Beispielhafte zugewiesene Kommentar-Topics


,comment_id,comment_text,topic_id,topic_label
1178,7541804977875550983,Wow I liked 💗,7,robot | expect | song
1192,7543754834214945557,Need this therapy too✨,3,health | sleep | worst
1197,7572978399367676686,and the little sister too,6,robot | sophia | polar
1198,7553098989831308087,yup that is me and my future wife...lol 🤣🤣,0,gay | animated | steal
1199,7553627849786852126,Is he bothering you queen?,3,health | sleep | worst


Beispielhafte Video-Level-Aggregation


,video_id,Typ,comment_count,topic_unique_labels,dominant_topic_label,video_engagement_rate
0,6857638022805032197,KI,194,11,gay | animated | steal,0.152487
1,6988896764241693958,KI,627,14,robot | cgi | generated,0.042539
2,7021870483289230593,KI,270,13,robot | expect | song,0.133174
3,7027954993026190598,KI,1451,14,robot | sophia | polar,0.107450
4,7034987201482001670,KI,19,8,gay | animated | steal,0.051886


In [5]:
comments_df.to_csv(COMMENT_RESULTS_CSV, index=False)
video_level_df.to_csv(VIDEO_LEVEL_CSV, index=False)
topic_engagement_df.to_csv(TOPIC_ENGAGEMENT_CSV, index=False)
topic_keywords_df.to_csv(TOPIC_KEYWORDS_CSV, index=False)
label_stopword_candidates_df.to_csv(LABEL_STOPWORD_CANDIDATES_CSV, index=False)
label_stopword_audit_df.to_csv(LABEL_STOPWORDS_CSV, index=False)
excluded_empty_topic_text_df.to_csv(TOPIC_TEXT_EXCLUDED_CSV, index=False)
missing_video_metadata_df.to_csv(MISSING_VIDEO_METADATA_CSV, index=False)
excluded_missing_engagement_df.to_csv(MISSING_ENGAGEMENT_EXCLUDED_CSV, index=False)
unresolved_video_metadata_df.to_csv(UNRESOLVED_VIDEO_METADATA_CSV, index=False)
engagement_source_video_df.to_csv(ENGAGEMENT_SOURCE_CSV, index=False)

saved_outputs = pd.DataFrame([
    COMMENT_RESULTS_CSV,
    VIDEO_LEVEL_CSV,
    TOPIC_ENGAGEMENT_CSV,
    TOPIC_KEYWORDS_CSV,
    LABEL_STOPWORD_CANDIDATES_CSV,
    LABEL_STOPWORDS_CSV,
    TOPIC_TEXT_EXCLUDED_CSV,
    MISSING_VIDEO_METADATA_CSV,
    MISSING_ENGAGEMENT_EXCLUDED_CSV,
    UNRESOLVED_VIDEO_METADATA_CSV,
    ENGAGEMENT_SOURCE_CSV,
], columns=["path"])
saved_outputs["file"] = saved_outputs["path"].map(lambda p: p.name)
saved_outputs["folder"] = saved_outputs["path"].map(lambda p: str(p.parent.relative_to(BASE_DIR)))

print(f"Themenanalyse gespeichert: {len(saved_outputs)} CSV-Dateien")
display(saved_outputs[["file", "folder"]])


Themenanalyse gespeichert: 11 CSV-Dateien


,file,folder
0,04_comment_topics_results.csv,comments/results
1,04_comment_topics_video_level.csv,comments/results
2,04_comment_topics_engagement_relations.csv,comments/results
3,04_comment_topics_keywords.csv,comments/results
4,04_comment_topics_label_stopword_candidates.csv,comments/results
5,04_comment_topics_label_stopwords.csv,comments/results
6,04_comment_topics_excluded_empty_topic_text.csv,comments/results
7,04_comment_topics_missing_video_metadata.csv,comments/results
8,04_comment_topics_excluded_missing_engagement.csv,comments/results
9,04_comment_topics_unresolved_video_metadata.csv,comments/results
